# Mean Absolute Error (MAE) for Regression Evaluation

Training a regression model is only half the task. The real question is:

How do we measure how good its predictions actually are?

For regression problems, evaluation is about measuring the gap between predicted values and true values consistently, honestly, and in a way that connects back to the real-world problem you're solving. One of the most intuitive and widely used metrics for this is Mean Absolute Error (MAE).

Unlike metrics that square errors or compute explained variance, MAE measures average prediction error in the same units as the target variable, making it directly interpretable by engineers, product managers, and business stakeholders alike.

If you are predicting house prices in lakhs and your MAE is 3.5, it means:

On average, your predictions are off by ₹3.5 lakhs.

Simple. Transparent. Actionable.

This lesson covers:

- What MAE is and how it is calculated
- Why MAE is intuitive and when that matters
- How it differs from MSE and RMSE, and when to choose each
- How to compute MAE in scikit-learn
- How to interpret MAE relative to a baseline
- How to use cross-validation with MAE
- Common pitfalls that lead to misinterpretation

## What Is Mean Absolute Error (MAE)?

Mean Absolute Error measures the average absolute difference between predicted and actual values across all samples:

$$
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|
$$

Where:

- `y_i` is the actual observed value
- `\hat{y}_i` is the model's predicted value
- `n` is the total number of samples

Two properties define MAE's behavior:

- Absolute value makes all errors positive, so over-predictions and under-predictions contribute equally, with no cancellation.
- No squaring means large errors are penalized proportionally, not quadratically. A prediction off by 10 contributes exactly 5x more than one off by 2.

This linearity is the key distinction between MAE and MSE-based metrics, and it drives almost every practical difference between them.

## Simple Example

Suppose we predict house prices in lakhs:

| Actual | Predicted | Error | Absolute Error |
| --- | --- | --- | --- |
| 50 | 47 | -3 | 3 |
| 60 | 65 | +5 | 5 |
| 40 | 42 | +2 | 2 |
| 70 | 68 | -2 | 2 |

Interpretation:

On average, predictions are off by ₹3 lakhs.

Notice that the raw errors (-3, +5, +2, -2) sum to +2, which means the model has a slight positive bias on this sample. MAE does not reveal directional bias; it only measures average magnitude. This is one reason why inspecting residuals alongside MAE is essential, not optional.

## Why MAE Is Intuitive

The appeal of MAE is that it answers a question anyone can understand:

On average, how far off are our predictions?

If you are predicting temperature in °C and MAE = 1.5, you know immediately that the forecast is wrong by 1.5 degrees on average. No transformation, no square roots, no mental gymnastics required.

Compare this to MSE. If MSE = 6.25 on the same task, the unit is degrees squared, which has no physical meaning. You have to take the square root to recover something interpretable: RMSE = 2.5°C.

MAE skips that step entirely. That makes it the natural choice whenever you need to communicate model performance to people who do not speak statistics.

## MAE vs MSE vs RMSE

Understanding the relationship between these three metrics is essential for choosing the right evaluation strategy for your problem.

| Metric | Formula | Error Penalty | Units | Outlier Sensitivity |
| --- | --- | --- | --- | --- |
| MAE | Mean of absolute errors | Linear | Same as target | Low |
| MSE | Mean of squared errors | Quadratic | Squared target units | High |
| RMSE | Square root of MSE | Quadratic | Same as target | High |

### Why the Penalty Type Matters

Consider four prediction errors: `[2, 2, 2, 20]`. The last value is an outlier, a single catastrophic prediction.

The outlier barely moves MAE, because it is just one value averaged with three small errors. But it dominates MSE and RMSE, because squaring transforms 20 into 400. This is the fundamental trade-off:

- MAE: every error counts equally. Three well-calibrated predictions and one catastrophic one still looks okay under the average.
- RMSE: the catastrophic prediction screams loudly. It dominates the metric, which is exactly what you want when large errors carry disproportionate consequences.

Neither behavior is wrong. They reflect different business realities. The question is whether your domain treats an error of 10 as roughly 5x worse than an error of 2, or much more than that.

## When to Use MAE

MAE is the right choice when:

- Interpretability is critical and stakeholders need error in concrete, real-world terms
- Outliers exist but should not dominate the primary evaluation metric
- Errors have roughly uniform cost, so being off by 10 is approximately twice as bad as being off by 5
- You are reporting model performance to non-technical audiences

Common applications include sales forecasting, demand prediction, inventory planning, delivery time estimation, energy consumption forecasting, and customer lifetime value estimation.

## When MAE May Not Be Ideal

MAE is the wrong primary metric when:

- Large errors are catastrophically costly, as in medical dosing, structural engineering, or financial risk
- You are using gradient-based optimization and want a smoother loss surface
- You want to penalize inconsistency, because MAE can hide occasional wild misses
- Errors may be systematically biased, because MAE does not distinguish random scatter from directional error

Inspect residuals alongside MAE so you do not miss bias or instability.

## Implementing MAE in Scikit-Learn

```python
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")
```

One line. Scikit-learn handles array validation and edge cases automatically. The function accepts array-like inputs such as NumPy arrays, pandas Series, or Python lists.

## Comparing Against a Baseline Using MAE

MAE in isolation is almost meaningless. Always compare your model against a mean baseline on the same test set.

```python
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

model = LinearRegression()
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_preds)
model_mae = mean_absolute_error(y_test, model_preds)
improvement = baseline_mae - model_mae
pct_improve = (improvement / baseline_mae) * 100
```

The model reduces average error by a concrete amount. That is the kind of result you can explain to a stakeholder without translation.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

# Built-in regression dataset with a real-valued target
X, y = load_diabetes(return_X_y=True, as_frame=True)

# Split first so the test set stays untouched until the end
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    )

# Baseline: predict the training mean
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

# Model: ordinary least squares regression
model = LinearRegression()
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_preds)
model_mae = mean_absolute_error(y_test, model_preds)
baseline_rmse = mean_squared_error(y_test, baseline_preds) ** 0.5
model_rmse = mean_squared_error(y_test, model_preds) ** 0.5
model_r2 = r2_score(y_test, model_preds)

improvement = baseline_mae - model_mae
pct_improve = (improvement / baseline_mae) * 100
mae_pct = (model_mae / y_test.mean()) * 100
residuals = model_preds - y_test.to_numpy()

cv_scores = -cross_val_score(
    model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error",
)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Model MAE: {model_mae:.2f}")
print(f"Improvement: {improvement:.2f} ({pct_improve:.1f}%)")
print(f"Model MAE as % of mean target: {mae_pct:.1f}%")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Model RMSE: {model_rmse:.2f}")
print(f"Model R^2: {model_r2:.3f}")
print(f"Residual mean (bias check): {residuals.mean():.2f}")
print(f"CV MAE per fold: {np.round(cv_scores, 2)}")
print(f"Mean CV MAE: {cv_scores.mean():.2f} +/- {cv_scores.std():.2f}")

## Interpreting MAE Properly

A MAE value without context is almost useless. Proper interpretation needs three anchors:

1. Target scale - What is the typical magnitude of the target variable?
2. Baseline performance - Is your model actually learning, or just marginally better than guessing?
3. Business tolerance - What level of error is acceptable for the decision being made?

## Using Cross-Validation with MAE

A single test-set evaluation can mislead, especially on smaller datasets. Cross-validation gives a more reliable picture of true generalization performance.

```python
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error",
)

mae_scores = -cv_scores
print(f"CV MAE per fold: {mae_scores.round(2)}")
print(f"Mean CV MAE: {mae_scores.mean():.3f}")
print(f"Std CV MAE: {mae_scores.std():.3f}")
```

Scikit-learn returns negative MAE because it uses a higher-is-better convention across scorers. Always flip the sign before reporting.

What to look for in CV results: a low mean MAE is good, but a low standard deviation across folds is equally important. A model with mean MAE = 7.5 ± 0.4 is far more trustworthy than one with mean MAE = 7.2 ± 3.1.

## Common Mistakes to Avoid

- Reporting MAE without a baseline
- Comparing MAE to RMSE as if they were directly interchangeable
- Forgetting to interpret MAE in target units after log transforms
- Ignoring directional bias in residuals
- Reporting only MAE when RMSE and R² are needed for context

## Practical Checklist Before Reporting MAE

- Computed on test data only
- Compared against a mean baseline MAE on the same test set
- Expressed as a percentage of mean target value for scale context
- Validated with cross-validation
- Residuals inspected for systematic bias
- Reported alongside RMSE and R²
- Interpreted against business tolerance

## Closing Reflection

Mean Absolute Error directly answers the question every honest practitioner should be asking: On average, how wrong are we?

It does not exaggerate errors. It does not hide interpretation behind squared units. It speaks the same language as the problem. If you are predicting in lakhs, MAE is in lakhs. If you are predicting in hours, MAE is in hours.

Train models carefully.
Evaluate honestly.
Interpret in context.


## Bonus Content

- Scikit-learn Model Evaluation Guide
- Understanding Regression Metrics - Scikit-learn